# 01: Clean results and pit stops
Goal: keep finishers only (results) and drop outlier stops (pit_stops)

In [3]:
import pandas as pd
from pathlib import Path

In [4]:
RAW = Path("../data/raw")
PROCESSED = Path("../data/processed")

print(RAW.exists())
print(sorted(p.name for p in RAW.iterdir()))

True
['.gitkeep', 'constructors.csv', 'drivers.csv', 'lap_times.csv', 'pit_stops.csv', 'races.csv', 'results.csv', 'status.csv']


In [5]:
status = pd.read_csv(RAW / "status.csv", na_values=r"\N")
results = pd.read_csv(RAW / "results.csv", na_values=r"\N")
pit_stops = pd.read_csv(RAW / "pit_stops.csv", na_values=r"\N")

In [6]:
status.head()

,statusId,status
0,1,Finished
1,2,Disqualified
2,3,Accident
3,4,Collision
4,5,Engine


In [7]:
status.shape

(139, 2)

In [8]:
status.info()

<class 'pandas.DataFrame'>
RangeIndex: 139 entries, 0 to 138
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   statusId  139 non-null    int64
 1   status    139 non-null    str  
dtypes: int64(1), str(1)
memory usage: 2.3 KB


In [9]:
FINISHED_STATUS_ID = 1

rows_in = len(results)
print("results row in:", rows_in)

results row in: 26759


In [10]:
results_status = results.merge(status, on="statusId", how="left", validate="many_to_one")

print("rows after join:", len(results_status))
print("row count unchanged:", len(results_status) == rows_in)

print("rows with no status text:", results_status["status"].isna().sum())

rows after join: 26759
row count unchanged: True
rows with no status text: 0


In [11]:
#keep only rows where status if "Finished"
is_finisher = (results_status["statusId"] == FINISHED_STATUS_ID)
results_clean = results_status[is_finisher].copy()

#summary of kept & dropped
rows_out = len(results_clean)
print("rows in: ", rows_in)
print("rows dropped:", rows_in - rows_out)
print("rows out: ", rows_out)
print("share kept: ", round(rows_out / rows_in, 3))

rows in:  26759
rows dropped: 19085
rows out:  7674
share kept:  0.287


In [12]:
print(results_clean["status"].unique())
print(results_clean[["position", "positionOrder"]].isna().sum())
print(results_clean[["position", "positionOrder"]].dtypes)

<StringArray>
['Finished']
Length: 1, dtype: str
position         0
positionOrder    0
dtype: int64
position         float64
positionOrder      int64
dtype: object


In [13]:
# Does the raw results file use the text \N for missing values?
print((results["position"] == r"\N").sum())

# Do any finishers still have it?
print((results_clean["position"] == r"\N").sum())

# Peek at the values
print(results_clean["position"].head())

0
0
0    1.0
1    2.0
2    3.0
3    4.0
4    5.0
Name: position, dtype: float64


In [14]:
test = pd.read_csv(RAW / "results.csv", na_values=r"\N")
print(test["position"].dtype)
print(test["position"].isna().sum())
print((test["position"] == r"\N").sum())

float64
10953
0


In [15]:
print(len(results), len(results_clean))
print(results["position"].isna().sum())
print((results_clean["position"] == results_clean["positionOrder"]).all())

26759 7674
10953
False


In [16]:
# Flag rows where the two columns disagree
mismatch = (results_clean["position"] != results_clean["positionOrder"])

# How many rows, and what share of finishers?
print(mismatch.sum(), round(mismatch.mean(), 4))

# Look at a sample of the disagreeing rows
cols = ["raceId", "driverId", "position", "positionOrder", "status"]
print(results_clean.loc[mismatch, cols].head(15))

4 0.0005
       raceId  driverId  position  positionOrder    status
10656     452       182       3.0              2  Finished
10657     452       172       4.0              3  Finished
10658     452       175       5.0              4  Finished
10659     452       176       6.0              5  Finished


In [17]:
# Every row for that race, from the unfiltered table, in order
cols = ["raceId", "driverId", "position", "positionOrder", "statusId", "status"]
race_452 = results_status.loc[results_status["raceId"] == 452, cols]
print(race_452.sort_values("positionOrder").head(10))

       raceId  driverId  position  positionOrder  statusId    status
10655     452       137       1.0              1         1  Finished
10656     452       182       3.0              2         1  Finished
10657     452       172       4.0              3         1  Finished
10658     452       175       5.0              4         1  Finished
10659     452       176       6.0              5         1  Finished
10660     452       117       7.0              6        11    +1 Lap
10661     452       118       8.0              7        11    +1 Lap
10662     452       192       9.0              8        11    +1 Lap
10663     452       163      10.0              9        11    +1 Lap
10664     452       193      11.0             10        11    +1 Lap


In [18]:
# Does position 2 exist anywhere in this race?
print(results_status.loc[(results_status["raceId"] == 452) & (results_status["position"] == 2)])

# Which race and year is this?
races = pd.read_csv(RAW / "races.csv", na_values=r"\N")
print(races.loc[races["raceId"] == 452, ["raceId", "year", "name", "date"]])

Empty DataFrame
Columns: [resultId, raceId, driverId, constructorId, number, grid, position, positionText, positionOrder, points, laps, time, milliseconds, fastestLap, rank, fastestLapTime, fastestLapSpeed, statusId, status]
Index: []
     raceId  year                  name        date
451     452  1983  Brazilian Grand Prix  1983-03-13


In [19]:
# Make sure the output folder exists, then save the cleaned table
PROCESSED.mkdir(parents=True, exist_ok=True)
results_clean.to_csv(PROCESSED / "results_clean.csv", index=False)

In [20]:
print(sorted(p.name for p in PROCESSED.iterdir()))
saved = pd.read_csv(PROCESSED / "results_clean.csv")
print(saved.shape)

['.gitkeep', 'pit_stops_clean.csv', 'results_clean.csv']
(7674, 19)


Clean pit_stops: remove outlier stops

Time Column: milliseconds

Rule: within each race, drop stops longer than 2x that race's median stop time. 


In [21]:
TIME_COL = "milliseconds"

print(pit_stops.dtypes)
print(pit_stops[["duration", "milliseconds"]].head(10))
print(pit_stops[["duration", "milliseconds"]].isna().sum())
print(pit_stops["milliseconds"].describe())

raceId          int64
driverId        int64
stop            int64
lap             int64
time              str
duration          str
milliseconds    int64
dtype: object
  duration  milliseconds
0   26.898         26898
1   25.021         25021
2   23.426         23426
3   23.251         23251
4   23.842         23842
5   23.643         23643
6   22.603         22603
7   24.863         24863
8   25.259         25259
9   25.342         25342
duration        0
milliseconds    0
dtype: int64
count    1.137100e+04
mean     8.523050e+04
std      3.104273e+05
min      1.289700e+04
25%      2.193750e+04
50%      2.360600e+04
75%      2.654400e+04
max      3.069017e+06
Name: milliseconds, dtype: float64


In [22]:
#does duration ever use minutes format
print(pit_stops["duration"].str.contains(":").sum())

plain = pit_stops[pit_stops["duration"].str.contains(":") == False]
print((plain["duration"].astype(float)*1000).round().eq(plain["milliseconds"]).all())

#which stops are extremes
print(pit_stops.sort_values("milliseconds").tail(10)[["raceId", "driverId", "stop", "lap", "milliseconds"]])

#which years covered
ps_years = pit_stops.merge(races[["raceId", "year"]], on="raceId", how="left", validate="many_to_one")
print(ps_years["year"].min(), ps_years["year"].max(), ps_years["year"].isna().sum())

517
True
      raceId  driverId  stop  lap  milliseconds
9145    1083         1     1    1       3046888
9153    1083       842     1    1       3048950
9150    1083       849     1    1       3051904
9147    1083         4     1    1       3055700
9151    1083       822     1    1       3055732
9152    1083       817     1    1       3063314
9154    1083       825     1    1       3063613
9157    1083       854     1    1       3065174
9155    1083        20     1    1       3067301
9156    1083       840     1    1       3069017
2011 2024 0


In [23]:
# Which race is it?
print(races.loc[races["raceId"] == 1083, ["raceId", "year", "name", "date"]])

# What does the race look like?
race_1083 = pit_stops[pit_stops["raceId"] == 1083]
print(len(race_1083))
print((race_1083["lap"] == 1).sum())
print(race_1083["milliseconds"].describe())

      raceId  year                name        date
1066    1083  2022  British Grand Prix  2022-07-03
47
17
count    4.700000e+01
mean     1.119664e+06
std      1.463701e+06
min      2.815500e+04
25%      2.889400e+04
50%      2.984600e+04
75%      3.036842e+06
max      3.069017e+06
Name: milliseconds, dtype: float64


In [24]:
OUTLIER_MULTIPLE = 2

stops = pit_stops.copy()
stops["race_median"] = stops.groupby("raceId")[TIME_COL].transform("median")
stops["cutoff"] = OUTLIER_MULTIPLE * stops["race_median"]

print(stops[["raceId", TIME_COL, "race_median", "cutoff"]].head())

   raceId  milliseconds  race_median   cutoff
0     841         26898      24181.0  48362.0
1     841         25021      24181.0  48362.0
2     841         23426      24181.0  48362.0
3     841         23251      24181.0  48362.0
4     841         23842      24181.0  48362.0


In [25]:
is_normal = (stops[TIME_COL]<=stops["cutoff"])

pit_stops_clean = stops[is_normal].copy()
pit_stops_dropped = stops[is_normal == False].copy()

rows_in = len(stops)
rows_out = len(pit_stops_clean)
print("rows in: ", rows_in)
print("rows dropped:", len(pit_stops_dropped))
print("rows out: ", rows_out)
print("share kept: ", round(rows_out/rows_in, 3))

rows in:  11371
rows dropped: 427
rows out:  10944
share kept:  0.962


In [26]:
print(pit_stops_dropped["raceId"].value_counts().head(10))
print(pit_stops_dropped[TIME_COL].describe())

raceId
1039    25
914     22
1045    19
1057    19
1061    19
1111    19
948     18
960     18
1080    18
1092    18
Name: count, dtype: int64
count    4.270000e+02
mean     1.258357e+06
std      7.304343e+05
min      3.686600e+04
25%      1.002958e+06
50%      1.302010e+06
75%      1.578879e+06
max      3.069017e+06
Name: milliseconds, dtype: float64


In [27]:
# For each race: total stops, dropped stops, and share dropped
by_race = pd.DataFrame({
    "stops": stops.groupby("raceId").size(),
    "dropped": pit_stops_dropped.groupby("raceId").size(),
}).fillna(0)
by_race["share_dropped"] = by_race["dropped"] / by_race["stops"]

# Races that lost the biggest share of their stops
print(by_race.sort_values("share_dropped", ascending=False).head(10))

# Medians of those races, to spot an inflated one
print(stops.groupby("raceId")[TIME_COL].median().sort_values().tail(5))

        stops  dropped  share_dropped
raceId                               
1117       38     18.0       0.473684
1061       41     19.0       0.463415
1038       37     17.0       0.459459
1141       35     16.0       0.457143
1092       42     18.0       0.428571
948        45     18.0       0.400000
1039       66     25.0       0.378788
1083       47     17.0       0.361702
960        51     18.0       0.352941
1080       53     18.0       0.339623
raceId
1141      58984.0
1100     879804.0
1072    1008569.0
967     1483133.5
1128    2369363.0
Name: milliseconds, dtype: float64


In [28]:
odd = [1141, 1100, 1072, 967, 1128]

# Which races are these?
print(races.loc[races["raceId"].isin(odd), ["raceId", "year", "name"]])

# For each: total stops, stops under 2 min (plausible real stops), stops over 10 min
for r in odd:
    s = stops.loc[stops["raceId"] == r, TIME_COL]
    print(r, len(s), (s < 120000).sum(), (s > 600000).sum())

# What the surviving table still contains
print(pit_stops_clean[TIME_COL].max())
print((pit_stops_clean[TIME_COL] > 120000).sum())

      raceId  year                      name
954      967  2016      Brazilian Grand Prix
1056    1072  2021  Saudi Arabian Grand Prix
1081    1100  2023     Australian Grand Prix
1108    1128  2024         Monaco Grand Prix
1121    1141  2024      São Paulo Grand Prix
1141 35 19 16
1100 65 19 46
1072 47 12 35
967 62 26 36
1128 23 7 16
2394053
121


In [29]:
# How many stops fall between 2 and 10 minutes, across all races?
in_between = stops[(stops[TIME_COL] > 120000) & (stops[TIME_COL] <= 600000)]
print(len(in_between))

# Distribution of everything over 1 minute, in minutes
print((stops.loc[stops[TIME_COL] > 60000, TIME_COL] / 60000).describe())
print(in_between["raceId"].value_counts().head())

21
count    517.000000
mean      22.688298
std       10.720325
min        1.002867
25%       17.026917
50%       21.886450
75%       26.388883
max       51.150283
Name: milliseconds, dtype: float64
raceId
1092    18
881      1
888      1
1044     1
Name: count, dtype: int64


ceiling for stops

In [30]:
# Stops longer than this multiple of their race's median are outliers
OUTLIER_MULTIPLE = 2

# Column with the stop time in numbers
TIME_COL = "milliseconds"

# Hard ceiling: anything longer is a stoppage wait (red flag etc.), not a real stop
MAX_STOP_MS = 120000

In [31]:
stops = pit_stops.copy()

# True for stops longer than the ceiling
stops["over_ceiling"] = (stops[TIME_COL] > MAX_STOP_MS)
print("stops over the ceiling:", stops["over_ceiling"].sum())

stops over the ceiling: 481


In [32]:
# Use only stops under the ceiling to compute each race's median
under = stops[stops["over_ceiling"] == False]
race_median = under.groupby("raceId")[TIME_COL].median()

# Attach each race's median to every stop in that race
# map() looks up each stop's raceId in race_median
stops["race_median"] = stops["raceId"].map(race_median)
stops["cutoff"] = OUTLIER_MULTIPLE * stops["race_median"]

print(stops[["raceId", TIME_COL, "race_median", "cutoff"]].head())

   raceId  milliseconds  race_median   cutoff
0     841         26898      24181.0  48362.0
1     841         25021      24181.0  48362.0
2     841         23426      24181.0  48362.0
3     841         23251      24181.0  48362.0
4     841         23842      24181.0  48362.0


In [33]:
# Keep a stop only if it is under the ceiling AND within 2x its race median
under_ceiling = (stops["over_ceiling"] == False)
within_cutoff = (stops[TIME_COL] <= stops["cutoff"])
is_normal = (under_ceiling & within_cutoff)

pit_stops_clean = stops[is_normal].copy()
pit_stops_dropped = stops[is_normal == False].copy()

# Summary, split by reason
n_ceiling = stops["over_ceiling"].sum()
n_multiple = (under_ceiling & (within_cutoff == False)).sum()

print("rows in:               ", len(stops))
print("dropped over ceiling:  ", n_ceiling)
print("dropped by 2x median:  ", n_multiple)
print("rows out:              ", len(pit_stops_clean))
print("share kept:            ", round(len(pit_stops_clean) / len(stops), 3))

rows in:                11371
dropped over ceiling:   481
dropped by 2x median:   74
rows out:               10816
share kept:             0.951


In [34]:
# Longest surviving stop (should be under the ceiling)
print(pit_stops_clean[TIME_COL].max())

# Stops over the ceiling that survived (should be 0)
print((pit_stops_clean[TIME_COL] > MAX_STOP_MS).sum())

# Rows add up (should print True)
print(len(pit_stops_clean) + len(pit_stops_dropped) == len(stops))

# Races that lost every stop
print(set(stops["raceId"]) - set(pit_stops_clean["raceId"]))

# Fewest surviving stops in any race (a tiny count makes a median shaky)
print(pit_stops_clean.groupby("raceId").size().sort_values().head())

# The races that failed before: how many stops do they keep now?
print(pit_stops_clean[pit_stops_clean["raceId"].isin([1100, 1072, 967, 1128])].groupby("raceId").size())

63280
0
True
set()
raceId
1128     7
1072    12
1100    16
926     16
1141    17
dtype: int64
raceId
967     26
1072    12
1100    16
1128     7
dtype: int64


In [35]:
# Remove helper columns so the saved table matches the raw pit_stops columns
pit_stops_out = pit_stops_clean.drop(columns=["over_ceiling", "race_median", "cutoff"])

PROCESSED.mkdir(parents=True, exist_ok=True)
pit_stops_out.to_csv(PROCESSED / "pit_stops_clean.csv", index=False)

# Reload and confirm
saved = pd.read_csv(PROCESSED / "pit_stops_clean.csv")
print(saved.shape, saved.columns.tolist())

(10816, 7) ['raceId', 'driverId', 'stop', 'lap', 'time', 'duration', 'milliseconds']


In [36]:
from f1_pipeline.transform import build_results_table, build_pit_stops_table

In [37]:
results_test = build_results_table(results, status)
pit_test = build_pit_stops_table(pit_stops)

rows in:      26759
rows dropped: 19085
rows out:     7674
share kept:   0.287
rows in:               11371
dropped over ceiling:  481
dropped by median x 2 : 74
rows out:              10816
share kept:            0.951


In [38]:
saved_results = pd.read_csv(PROCESSED / "results_clean.csv", na_values=r"\N")
saved_pits = pd.read_csv(PROCESSED / "pit_stops_clean.csv", na_values=r"\N")

print(results_test.shape, saved_results.shape)
print(pit_test.shape, saved_pits.shape)

(7674, 19) (7674, 19)
(10816, 7) (10816, 7)


In [39]:
# Same values, ignoring row numbers? (should print True)
print(results_test.reset_index(drop=True).equals(saved_results))
print(pit_test.reset_index(drop=True).equals(saved_pits))

False
True
